# NB8 Mini — Forensic Audit of Linear-Probe Results (NB4 v2 / NB7 TPU)

**Goal:** investigate the suspiciously strong NB8 probe results without rerunning the full hidden-state sweep, re-extracting representations, or loading either model.

This mini notebook audits the current authoritative **2,001-row NB4 v2 dataset** and the rebuilt NB8 outputs. It tests whether explicit recovery-state cues in `state_text` make reversibility lexically predictable.

It runs four lightweight TF-IDF + logistic-regression baselines:
1. `action_text_only`
2. `state_text_only`
3. `masked_state_text_only`
4. `masked_state_plus_action`

Each is evaluated for binary and three-class labels, H1/H2/H3, and all three frozen split seeds (72 fits total). It also compares these baselines with existing NB8 probe results, without refitting the hidden-state probes.

**Scope note:** the older forensic report about 2,052-row legacy representations and layer-0 collisions is a separate investigation. This notebook is pinned to the newer 2,001-row NB4 v2 / NB7 TPU run and does not mix those legacy findings into the current results.

Inputs are read-only. Outputs are saved in a new audit folder and mirrored to both Drive roots.

In [1]:
# 1. Mount Drive and define paths
from pathlib import Path
import os, json, hashlib, re, shutil, sys, time
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive")
IR = DRIVE / "ir"
IR_FINAL = DRIVE / "ir_final"
SNAPSHOT = "NB4_v2_snapshot_20261123T130007318638Z"

DATASET_PATH = IR / SNAPSHOT / "integrated_dataset_v2.jsonl"
SPLITS_PATH = IR / SNAPSHOT / "grouped_splits_v2.json"
NB8_DIR = IR / "NB8_linear_probes"
NB8_RESULTS = NB8_DIR / "nb8_probe_results.csv"
NB8_AGGREGATE = NB8_DIR / "nb8_probe_aggregate.csv"

OUT = IR / "NB8_mini_forensic_audit_20261124"
OUT.mkdir(parents=True, exist_ok=True)

EXPECTED_SHA256 = "0d890a39de3953e1a4d8f03d0cde7fad9a27ebba143e07c0b31482d8bb13c9bd"
EXPECTED_TOTAL, EXPECTED_PRIMARY = 2001, 2000
SEEDS, HORIZONS = [0, 1, 2], ["1", "2", "3"]
LABELS_3 = ["reversible", "partially_reversible", "irreversible"]
LABELS_2 = ["reversible", "irreversible"]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("Python:", sys.version)
print("Dataset:", DATASET_PATH, DATASET_PATH.is_file())
print("Splits:", SPLITS_PATH, SPLITS_PATH.is_file())
print("NB8 results:", NB8_RESULTS.is_file())
print("NB8 aggregate:", NB8_AGGREGATE.is_file())
print("Audit output:", OUT)


Mounted at /content/drive
Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Dataset: /content/drive/MyDrive/ir/NB4_v2_snapshot_20261123T130007318638Z/integrated_dataset_v2.jsonl True
Splits: /content/drive/MyDrive/ir/NB4_v2_snapshot_20261123T130007318638Z/grouped_splits_v2.json True
NB8 results: True
NB8 aggregate: True
Audit output: /content/drive/MyDrive/ir/NB8_mini_forensic_audit_20261124


In [2]:
# 2. Load and validate dataset + frozen grouped splits
assert DATASET_PATH.is_file(), f"Missing dataset: {DATASET_PATH}"
assert SPLITS_PATH.is_file(), f"Missing splits: {SPLITS_PATH}"

dataset_sha256 = sha256_file(DATASET_PATH)
print("Dataset SHA-256:", dataset_sha256)
assert dataset_sha256 == EXPECTED_SHA256, "Dataset hash mismatch; stop."

all_records = [json.loads(x) for x in DATASET_PATH.read_text(encoding="utf-8").splitlines() if x.strip()]
assert len(all_records) == EXPECTED_TOTAL

def meta_of(r):
    return r.get("meta") or r.get("metadata") or {}

def horizon_labels(r):
    m = meta_of(r)
    return m.get("labels_by_horizon") or r.get("labels_by_horizon") or {}

def pair_id_of(r):
    return r.get("pair_id", meta_of(r).get("pair_id"))

def scenario_id_of(r):
    return r.get("scenario_id", meta_of(r).get("scenario_id"))

def is_primary(r):
    env = str(r.get("environment", "")).lower()
    role = str(r.get("dataset_role", meta_of(r).get("dataset_role", ""))).lower()
    return env in {"python_sandbox", "sandbox", "python sandbox"} or role == "primary_sandbox"

primary = [r for r in all_records if is_primary(r)]
non_primary = [r for r in all_records if not is_primary(r)]
assert len(primary) == EXPECTED_PRIMARY, (
    f"Expected 2,000 primary Sandbox rows; got {len(primary)}. "
    "Inspect environment/role fields before continuing."
)

for i, r in enumerate(primary):
    for k in ("id", "state_text", "action_text", "label"):
        assert k in r and r[k] is not None, f"Row {i}: missing {k}"
    assert r["label"] in LABELS_3
    assert isinstance(r["state_text"], str) and isinstance(r["action_text"], str)
    for h in HORIZONS:
        assert horizon_labels(r).get(h) in LABELS_3, (r["id"], h, horizon_labels(r).get(h))

all_ids = [str(r["id"]) for r in all_records]
assert len(all_ids) == len(set(all_ids))
primary_by_id = {str(r["id"]): r for r in primary}
primary_ids = set(primary_by_id)

split_payload = json.loads(SPLITS_PATH.read_text(encoding="utf-8"))
assert split_payload.get("dataset_sha256") == dataset_sha256
SPLITS = {}

for seed in SEEDS:
    sp = split_payload["splits"][str(seed)]
    tr, te = set(map(str, sp["train_ids"])), set(map(str, sp["test_ids"]))
    assert tr.isdisjoint(te) and tr | te == set(all_ids)
    trp, tep = tr & primary_ids, te & primary_ids
    assert trp.isdisjoint(tep) and trp | tep == primary_ids

    tr_scen = {scenario_id_of(primary_by_id[i]) for i in trp}
    te_scen = {scenario_id_of(primary_by_id[i]) for i in tep}
    assert not (tr_scen & te_scen), f"Scenario leakage seed {seed}"

    tr_pairs = {pair_id_of(primary_by_id[i]) for i in trp if pair_id_of(primary_by_id[i]) is not None}
    te_pairs = {pair_id_of(primary_by_id[i]) for i in tep if pair_id_of(primary_by_id[i]) is not None}
    assert not (tr_pairs & te_pairs), f"Pair leakage seed {seed}"

    SPLITS[seed] = {"train_ids": sorted(trp), "test_ids": sorted(tep)}
    print(f"Seed {seed}: train={len(trp)}, test={len(tep)}, scenario/pair leakage=0")

print("Integrated:", len(all_records), "| Primary Sandbox:", len(primary), "| Excluded:", len(non_primary))
print("Primary labels:", dict(Counter(r["label"] for r in primary)))
print("Dataset and split validation: PASS")


Dataset SHA-256: 0d890a39de3953e1a4d8f03d0cde7fad9a27ebba143e07c0b31482d8bb13c9bd
Seed 0: train=1598, test=402, scenario/pair leakage=0
Seed 1: train=1600, test=400, scenario/pair leakage=0
Seed 2: train=1600, test=400, scenario/pair leakage=0
Integrated: 2001 | Primary Sandbox: 2000 | Excluded: 1
Primary labels: {'reversible': 750, 'irreversible': 1000, 'partially_reversible': 250}
Dataset and split validation: PASS


In [3]:
# 3. Audit the explicit recovery cues already identified in NB8
CUE_PATTERNS = {
    "version_history_available": re.compile(r"version_history_available\\s*[:=]\\s*(true|false)", re.I),
    "recycle_bin_enabled": re.compile(r"recycle_bin_enabled\\s*[:=]\\s*(true|false)", re.I),
    "recall_window_open": re.compile(r"recall_window_open\\s*[:=]\\s*(true|false)", re.I),
    "refund_eligible": re.compile(r"refund_eligible\\s*[:=]\\s*(true|false)", re.I),
    "transfer_status": re.compile(r"transfer_status\\s*[:=]\\s*[A-Za-z_ -]+", re.I),
}

cue_rows = []
for r in primary:
    found = [name for name, pat in CUE_PATTERNS.items() if pat.search(r["state_text"])]
    row = {
        "id": str(r["id"]),
        "label": r["label"],
        "pair_id": pair_id_of(r),
        "scenario_id": scenario_id_of(r),
        "cue_count": len(found),
        "cue_names": "|".join(found),
    }
    for h in HORIZONS:
        row[f"label_H{h}"] = horizon_labels(r)[h]
    for name in CUE_PATTERNS:
        row[f"has_{name}"] = name in found
    cue_rows.append(row)

cue_df = pd.DataFrame(cue_rows)
cue_summary_rows = []
for cue in CUE_PATTERNS:
    for label in LABELS_3:
        sub = cue_df[cue_df["label"] == label]
        cue_summary_rows.append({
            "cue": cue, "label": label, "n": len(sub),
            "present": int(sub[f"has_{cue}"].sum()),
            "prevalence": float(sub[f"has_{cue}"].mean()) if len(sub) else np.nan
        })
cue_summary_df = pd.DataFrame(cue_summary_rows)
display(cue_summary_df)
print("Rows with >=1 detected cue:", int((cue_df.cue_count > 0).sum()), "/", len(cue_df))


,cue,label,n,present,prevalence
0,version_history_available,reversible,750,0,0.0
1,version_history_available,partially_reversible,250,0,0.0
2,version_history_available,irreversible,1000,0,0.0
3,recycle_bin_enabled,reversible,750,0,0.0
4,recycle_bin_enabled,partially_reversible,250,0,0.0
5,recycle_bin_enabled,irreversible,1000,0,0.0
6,recall_window_open,reversible,750,0,0.0
7,recall_window_open,partially_reversible,250,0,0.0
8,recall_window_open,irreversible,1000,0,0.0
9,refund_eligible,reversible,750,0,0.0


Rows with >=1 detected cue: 0 / 2000


In [4]:

# ============================================================
# 4. BUILD TEXT VIEWS — CORRECTED RECOVERY-CUE MASKING
# ============================================================

import re

RECOVERY_CUE_PATTERNS = {
    "version_history_available":
        r"\bversion_history_available\s*=\s*(?:true|false)",
    "recycle_bin_enabled":
        r"\brecycle_bin_enabled\s*=\s*(?:true|false)",
    "recall_window_open":
        r"\brecall_window_open\s*=\s*(?:true|false)",
    "refund_eligible":
        r"\brefund_eligible\s*=\s*(?:true|false)",
    "transfer_status":
        r"\btransfer_status\s*=\s*[\w-]+",
}

def mask_recovery_cues(text):
    text = str(text or "")

    for pattern in RECOVERY_CUE_PATTERNS.values():
        text = re.sub(
            pattern,
            "[RECOVERY_CUE_MASKED]",
            text,
            flags=re.IGNORECASE
        )

    return text

TEXT_VIEWS = {
    "action_text_only": {},
    "state_text_only": {},
    "masked_state_text_only": {},
    "masked_state_plus_action": {},
}

cue_counts = {name: 0 for name in RECOVERY_CUE_PATTERNS}

for record_id, rec in primary_by_id.items():

    action = str(rec.get("action_text", "") or "")
    state = str(rec.get("state_text", "") or "")

    masked_state = mask_recovery_cues(state)

    TEXT_VIEWS["action_text_only"][record_id] = action
    TEXT_VIEWS["state_text_only"][record_id] = state
    TEXT_VIEWS["masked_state_text_only"][record_id] = masked_state
    TEXT_VIEWS["masked_state_plus_action"][record_id] = (
        masked_state + " " + action
    )

    for name, pattern in RECOVERY_CUE_PATTERNS.items():
        if re.search(pattern, state, flags=re.IGNORECASE):
            cue_counts[name] += 1

# Validate all four views
assert all(len(view) == len(primary_by_id) for view in TEXT_VIEWS.values())

# Confirm masking actually changed affected records
changed = sum(
    TEXT_VIEWS["state_text_only"][rid] !=
    TEXT_VIEWS["masked_state_text_only"][rid]
    for rid in primary_by_id
)

assert changed > 0, "Masking did not change any records."

print("TEXT VIEWS REBUILT")
print("Records per view:", len(primary_by_id))
print("Records changed by masking:", changed)
print("\nDetected recovery cues:")

for name, count in cue_counts.items():
    print(f"{name}: {count}")

print("\nSample masking verification:")
sample_id = next(
    rid for rid in primary_by_id
    if TEXT_VIEWS["state_text_only"][rid] !=
       TEXT_VIEWS["masked_state_text_only"][rid]
)

print("Original:", TEXT_VIEWS["state_text_only"][sample_id])
print("Masked:  ", TEXT_VIEWS["masked_state_text_only"][sample_id])

for name, pattern in RECOVERY_CUE_PATTERNS.items():
    assert not re.search(
        pattern,
        TEXT_VIEWS["masked_state_text_only"][sample_id],
        flags=re.IGNORECASE
    ), f"Recovery cue remains: {name}"

print("\nMasking validation: PASS")

TEXT VIEWS REBUILT
Records per view: 2000
Records changed by masking: 2000

Detected recovery cues:
version_history_available: 500
recycle_bin_enabled: 500
recall_window_open: 500
refund_eligible: 500
transfer_status: 500

Sample masking verification:
Original: Simulated edit operation. Target=object-00000; original_value=original-00000; changed_value=changed-00000; amount=100; version_history_available=True.
Masked:   Simulated edit operation. Target=object-00000; original_value=original-00000; changed_value=changed-00000; amount=100; [RECOVERY_CUE_MASKED].

Masking validation: PASS


In [5]:

# ============================================================
# 5. TF-IDF + LOGISTIC REGRESSION — CORRECTED FUNCTION
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)

def fit_evaluate_text_view(
    train_ids, test_ids, horizon, task, text_lookup
):
    train_recs = [primary_by_id[i] for i in train_ids]
    test_recs = [primary_by_id[i] for i in test_ids]

    ytr_all = np.asarray(
        [horizon_labels(r)[horizon] for r in train_recs],
        dtype=object
    )
    yte_all = np.asarray(
        [horizon_labels(r)[horizon] for r in test_recs],
        dtype=object
    )

    if task == "binary":
        tr_keep = np.isin(ytr_all, LABELS_2)
        te_keep = np.isin(yte_all, LABELS_2)

        ytr = ytr_all[tr_keep]
        yte = yte_all[te_keep]

        tr_ids = np.asarray(train_ids)[tr_keep].tolist()
        te_ids = np.asarray(test_ids)[te_keep].tolist()

        labels = LABELS_2

    else:
        ytr, yte = ytr_all, yte_all
        tr_ids, te_ids = list(train_ids), list(test_ids)
        labels = LABELS_3

    if set(ytr) != set(labels) or set(yte) != set(labels):
        raise ValueError(
            f"Missing classes: task={task}, horizon={horizon}"
        )

    xtr = [text_lookup[i] for i in tr_ids]
    xte = [text_lookup[i] for i in te_ids]

    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        max_features=30000,
        sublinear_tf=True,
        token_pattern=r"(?u)\b\w+\b"
    )

    Xtr = vectorizer.fit_transform(xtr)
    Xte = vectorizer.transform(xte)

    clf = LogisticRegression(
        C=1.0,
        solver="lbfgs",
        l1_ratio=0,
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )

    clf.fit(Xtr, ytr)
    pred = clf.predict(Xte)

    return {
        "n_train": len(ytr),
        "n_test": len(yte),
        "vocabulary_size": len(vectorizer.vocabulary_),
        "accuracy": accuracy_score(yte, pred),
        "balanced_accuracy": balanced_accuracy_score(yte, pred),
        "macro_f1": f1_score(
            yte, pred, labels=labels,
            average="macro", zero_division=0
        ),
        "confusion_matrix": confusion_matrix(
            yte, pred, labels=labels
        ).tolist()
    }

print("Corrected TF-IDF baseline function ready.")

Corrected TF-IDF baseline function ready.


In [6]:
baseline_rows = []

started = time.time()

for view_name, text_lookup in TEXT_VIEWS.items():

    print(f"\n{'=' * 65}")
    print("TEXT VIEW:", view_name)
    print("=" * 65)

    for horizon in HORIZONS:

        for seed in SEEDS:

            split = SPLITS[seed]

            for task in ("binary", "3class"):

                metrics = fit_evaluate_text_view(
                    split["train_ids"],
                    split["test_ids"],
                    horizon,
                    task,
                    text_lookup
                )

                baseline_rows.append({
                    "view": view_name,
                    "horizon": horizon,
                    "seed": seed,
                    "task": task,
                    **metrics
                })

                print(
                    f"H{horizon} | Seed {seed} | {task} | "
                    f"Accuracy={metrics['accuracy']:.4f} | "
                    f"Balanced Accuracy={metrics['balanced_accuracy']:.4f} | "
                    f"Macro-F1={metrics['macro_f1']:.4f}"
                )

# Create results DataFrame
text_results = pd.DataFrame(baseline_rows)

# Validate expected configuration count
assert len(text_results) == 72, (
    f"Expected 72 configurations, got {len(text_results)}"
)

expected_views = set(TEXT_VIEWS.keys())
assert set(text_results["view"]) == expected_views
assert set(text_results["horizon"].astype(str)) == set(HORIZONS)
assert set(text_results["seed"]) == set(SEEDS)
assert set(text_results["task"]) == {"binary", "3class"}

print("\n" + "=" * 65)
print("TEXT BASELINE EXPERIMENT COMPLETE")
print("=" * 65)

print("Configurations completed:", len(text_results), "/ 72")
print("Elapsed minutes:", round((time.time() - started) / 60, 2))

display(text_results.head(12))


TEXT VIEW: action_text_only
H1 | Seed 0 | binary | Accuracy=0.3136 | Balanced Accuracy=0.6142 | Macro-F1=0.3077


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


H1 | Seed 0 | 3class | Accuracy=0.1393 | Balanced Accuracy=0.6667 | Macro-F1=0.1637
H1 | Seed 1 | binary | Accuracy=0.3264 | Balanced Accuracy=0.6079 | Macro-F1=0.3250
H1 | Seed 1 | 3class | Accuracy=0.1775 | Balanced Accuracy=0.6667 | Macro-F1=0.2063


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


H1 | Seed 2 | binary | Accuracy=0.3143 | Balanced Accuracy=0.6106 | Macro-F1=0.3104
H1 | Seed 2 | 3class | Accuracy=0.1525 | Balanced Accuracy=0.6667 | Macro-F1=0.1814
H2 | Seed 0 | binary | Accuracy=0.4427 | Balanced Accuracy=0.6186 | Macro-F1=0.4375


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


... (60 outputs left out)

H3 | Seed 1 | 3class | Accuracy=0.5000 | Balanced Accuracy=0.6667 | Macro-F1=0.4444
H3 | Seed 2 | binary | Accuracy=0.5634 | Balanced Accuracy=0.6125 | Macro-F1=0.5170


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


H3 | Seed 2 | 3class | Accuracy=0.5000 | Balanced Accuracy=0.6667 | Macro-F1=0.4444

TEXT BASELINE EXPERIMENT COMPLETE
Configurations completed: 72 / 72
Elapsed minutes: 0.31


,view,horizon,seed,task,n_train,n_test,vocabulary_size,accuracy,balanced_accuracy,macro_f1,confusion_matrix
0,action_text_only,1,0,binary,1527,389,1618,0.313625,0.614162,0.307695,"[[43, 0], [267, 79]]"
1,action_text_only,1,0,3class,1598,402,1618,0.139303,0.666667,0.163748,"[[43, 0, 0], [0, 13, 0], [267, 79, 0]]"
2,action_text_only,1,1,binary,1533,383,1598,0.326371,0.607903,0.325041,"[[54, 0], [258, 71]]"
3,action_text_only,1,1,3class,1600,400,1598,0.177500,0.666667,0.206297,"[[54, 0, 0], [0, 17, 0], [258, 71, 0]]"
4,action_text_only,1,2,binary,1531,385,1624,0.314286,0.610619,0.310373,"[[46, 0], [264, 75]]"
5,action_text_only,1,2,3class,1600,400,1624,0.152500,0.666667,0.181380,"[[46, 0, 0], [0, 15, 0], [264, 75, 0]]"
6,action_text_only,2,0,binary,1458,375,1618,0.442667,0.618613,0.437483,"[[101, 0], [209, 65]]"
7,action_text_only,2,0,3class,1598,402,1618,0.318408,0.666667,0.315089,"[[101, 0, 0], [0, 27, 0], [209, 65, 0]]"
8,action_text_only,2,1,binary,1466,367,1598,0.438692,0.605364,0.427639,"[[106, 0], [206, 55]]"
9,action_text_only,2,1,3class,1600,400,1598,0.347500,0.666667,0.350877,"[[106, 0, 0], [0, 33, 0], [206, 55, 0]]"


In [7]:
# 7. Aggregate baseline metrics across the three frozen split seeds
text_aggregate = (
    text_results.groupby(["view", "horizon", "task"], as_index=False)
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_sd=("accuracy", "std"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_sd=("balanced_accuracy", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_sd=("macro_f1", "std"),
        n_seeds=("seed", "nunique")
    )
)
display(text_aggregate.sort_values(["task", "horizon", "balanced_accuracy_mean"], ascending=[True, True, False]))


,view,horizon,task,accuracy_mean,accuracy_sd,balanced_accuracy_mean,balanced_accuracy_sd,macro_f1_mean,macro_f1_sd,n_seeds
18,state_text_only,1,3class,0.656434,0.019400,0.864311,0.004577,0.570841,0.036496,3
0,action_text_only,1,3class,0.156434,0.019400,0.666667,0.000000,0.183809,0.021378,3
6,masked_state_plus_action,1,3class,0.214768,0.079348,0.645868,0.018217,0.230845,0.068589,3
12,masked_state_text_only,1,3class,0.214768,0.079348,0.645868,0.018217,0.230845,0.068589,3
20,state_text_only,2,3class,0.829469,0.015749,0.915319,0.005914,0.815319,0.026940,3
2,action_text_only,2,3class,0.329469,0.015749,0.666667,0.000000,0.331334,0.018121,3
8,masked_state_plus_action,2,3class,0.329469,0.015749,0.666667,0.000000,0.331334,0.018121,3
14,masked_state_text_only,2,3class,0.329469,0.015749,0.666667,0.000000,0.331334,0.018121,3
22,state_text_only,3,3class,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,3
4,action_text_only,3,3class,0.500000,0.000000,0.666667,0.000000,0.444444,0.000000,3


In [8]:
# 8. Load existing NB8 hidden-state results; do not refit any probe
probe_results = pd.read_csv(NB8_RESULTS) if NB8_RESULTS.is_file() else None
probe_aggregate = pd.read_csv(NB8_AGGREGATE) if NB8_AGGREGATE.is_file() else None

print("Per-configuration results available:", probe_results is not None)
print("Aggregate results available:", probe_aggregate is not None)

final_rows = None
if probe_results is not None:
    print("Probe rows:", len(probe_results))
    print("Duplicate result keys:", int(probe_results["result_key"].duplicated().sum()))
    print("Models:", sorted(probe_results["model"].astype(str).unique()))

if probe_aggregate is not None:
    # Fixed final layer only; no best-layer selection on test scores.
    final_layers = {"qwen": 28, "llama": 32}
    final_rows = probe_aggregate[
        probe_aggregate.apply(
            lambda r: int(r["layer_index"]) == final_layers.get(str(r["model"]).lower(), -999),
            axis=1
        )
    ].copy()
    display(final_rows.sort_values(["model", "task", "horizon", "representation"]))


Per-configuration results available: True
Aggregate results available: True
Probe rows: 2232
Duplicate result keys: 0
Models: ['llama', 'qwen']


,model,representation,layer_index,horizon,task,balanced_accuracy_mean,balanced_accuracy_sd,macro_f1_mean,macro_f1_sd,accuracy_mean,auroc_mean,n_seeds
192,llama,last_action_token,32,1,3class,0.762137,0.041017,0.672105,0.017342,0.848586,0.934942,3
390,llama,mean_action_tokens,32,1,3class,0.748552,0.053255,0.663962,0.059244,0.841136,0.926766,3
194,llama,last_action_token,32,2,3class,0.793326,0.033378,0.741470,0.028596,0.766231,0.892469,3
392,llama,mean_action_tokens,32,2,3class,0.759776,0.006099,0.708569,0.023630,0.727148,0.867177,3
196,llama,last_action_token,32,3,3class,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,3
394,llama,mean_action_tokens,32,3,3class,0.998571,0.001238,0.998729,0.001101,0.998333,1.000000,3
193,llama,last_action_token,32,1,binary,0.846304,0.019447,0.809237,0.009065,0.908383,0.939067,3
391,llama,mean_action_tokens,32,1,binary,0.787424,0.024334,0.764722,0.032639,0.890303,0.925490,3
195,llama,last_action_token,32,2,binary,0.783000,0.017247,0.766145,0.018340,0.803079,0.867064,3
393,llama,mean_action_tokens,32,2,binary,0.716177,0.029463,0.707337,0.023504,0.758044,0.828546,3


In [9]:
# 9. Put text baselines beside fixed-final-layer hidden-state probes
comparison_rows = []
for _, r in text_aggregate.iterrows():
    comparison_rows.append({
        "source": "TFIDF_text",
        "model": "not_applicable",
        "view_or_representation": r["view"],
        "layer_index": None,
        "horizon": str(r["horizon"]),
        "task": r["task"],
        "balanced_accuracy_mean": r["balanced_accuracy_mean"],
        "balanced_accuracy_sd": r["balanced_accuracy_sd"],
        "accuracy_mean": r["accuracy_mean"]
    })

if final_rows is not None:
    for _, r in final_rows.iterrows():
        comparison_rows.append({
            "source": "hidden_state_fixed_final_layer",
            "model": r["model"],
            "view_or_representation": r["representation"],
            "layer_index": int(r["layer_index"]),
            "horizon": str(r["horizon"]),
            "task": r["task"],
            "balanced_accuracy_mean": r["balanced_accuracy_mean"],
            "balanced_accuracy_sd": r["balanced_accuracy_sd"],
            "accuracy_mean": r["accuracy_mean"]
        })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.sort_values(["task", "horizon", "source", "model"]))


,source,model,view_or_representation,layer_index,horizon,task,balanced_accuracy_mean,balanced_accuracy_sd,accuracy_mean
0,TFIDF_text,not_applicable,action_text_only,NaN,1,3class,0.666667,0.000000,0.156434
6,TFIDF_text,not_applicable,masked_state_plus_action,NaN,1,3class,0.645868,0.018217,0.214768
12,TFIDF_text,not_applicable,masked_state_text_only,NaN,1,3class,0.645868,0.018217,0.214768
18,TFIDF_text,not_applicable,state_text_only,NaN,1,3class,0.864311,0.004577,0.656434
24,hidden_state_fixed_final_layer,llama,last_action_token,32.0,1,3class,0.762137,0.041017,0.848586
30,hidden_state_fixed_final_layer,llama,mean_action_tokens,32.0,1,3class,0.748552,0.053255,0.841136
36,hidden_state_fixed_final_layer,qwen,last_action_token,28.0,1,3class,0.586426,0.030041,0.740452
42,hidden_state_fixed_final_layer,qwen,mean_action_tokens,28.0,1,3class,0.599576,0.023623,0.761211
2,TFIDF_text,not_applicable,action_text_only,NaN,2,3class,0.666667,0.000000,0.329469
8,TFIDF_text,not_applicable,masked_state_plus_action,NaN,2,3class,0.666667,0.000000,0.329469


## Interpretation guardrails

- If `state_text_only` is very strong, state text alone carries substantial label information.
- If `masked_state_text_only` falls sharply, the five detected recovery flags explain a meaningful part of that lexical predictability.
- If `action_text_only` is weak, tool/action wording alone is insufficient.
- If `masked_state_plus_action` remains strong, other lexical or structural cues may remain.
- Compare with fixed final-layer probe rows; do not choose a best layer using test-set performance and report it as unbiased.

These text baselines quantify what a simple lexical classifier can recover. They do **not**, by themselves, prove that the hidden-state probes rely on exactly the same cues. A cue-masked hidden-state probe is a separate, targeted follow-up.

The five cue families are those already identified in NB8 diagnostics: `version_history_available`, `recycle_bin_enabled`, `recall_window_open`, `refund_eligible`, and `transfer_status`. Document any newly discovered cue separately rather than silently adding it after seeing test results.

In [10]:
# 10. Save outputs and compact report
text_results_path = OUT / "nb8_mini_text_baseline_by_seed.csv"
text_aggregate_path = OUT / "nb8_mini_text_baseline_aggregate.csv"
cue_path = OUT / "nb8_mini_recovery_cue_audit.csv"
cue_summary_path = OUT / "nb8_mini_recovery_cue_summary.csv"
comparison_path = OUT / "nb8_mini_text_vs_probe_comparison.csv"

text_results.to_csv(text_results_path, index=False)
text_aggregate.to_csv(text_aggregate_path, index=False)
cue_df.to_csv(cue_path, index=False)
cue_summary_df.to_csv(cue_summary_path, index=False)
comparison_df.to_csv(comparison_path, index=False)

def h3_binary_row(view):
    q = text_aggregate[
        (text_aggregate["view"] == view) &
        (text_aggregate["horizon"].astype(str) == "3") &
        (text_aggregate["task"] == "binary")
    ]
    return None if q.empty else q.iloc[0].to_dict()

report = {
    "audit": "NB8 Mini Forensic Audit — current NB4 v2 / NB7 TPU",
    "dataset_sha256": dataset_sha256,
    "integrated_records": len(all_records),
    "primary_sandbox_records": len(primary),
    "excluded_non_primary_records": len(non_primary),
    "split_seeds": SEEDS,
    "horizons": HORIZONS,
    "text_views": list(TEXT_VIEWS),
    "text_baseline_fits": len(text_results),
    "all_72_fits_completed": len(text_results) == 72,
    "records_with_at_least_one_detected_recovery_cue": int((cue_df.cue_count > 0).sum()),
    "fraction_with_at_least_one_detected_recovery_cue": float((cue_df.cue_count > 0).mean()),
    "h3_binary_baselines": {v: h3_binary_row(v) for v in TEXT_VIEWS},
    "existing_nb8_probe_results_available": NB8_RESULTS.is_file(),
    "existing_nb8_aggregate_available": NB8_AGGREGATE.is_file(),
    "outputs": {
        "text_results": str(text_results_path),
        "text_aggregate": str(text_aggregate_path),
        "cue_audit": str(cue_path),
        "cue_summary": str(cue_summary_path),
        "comparison": str(comparison_path)
    },
    "interpretation_warning": (
        "Lexical baselines quantify input-level predictability; they do not alone "
        "establish that hidden-state probes use the same cues."
    )
}
report_path = OUT / "nb8_mini_forensic_audit_report.json"
report_path.write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")
print(json.dumps(report, indent=2, default=str))


{
  "audit": "NB8 Mini Forensic Audit \u2014 current NB4 v2 / NB7 TPU",
  "dataset_sha256": "0d890a39de3953e1a4d8f03d0cde7fad9a27ebba143e07c0b31482d8bb13c9bd",
  "integrated_records": 2001,
  "primary_sandbox_records": 2000,
  "excluded_non_primary_records": 1,
  "split_seeds": [
    0,
    1,
    2
  ],
  "horizons": [
    "1",
    "2",
    "3"
  ],
  "text_views": [
    "action_text_only",
    "state_text_only",
    "masked_state_text_only",
    "masked_state_plus_action"
  ],
  "text_baseline_fits": 72,
  "all_72_fits_completed": true,
  "records_with_at_least_one_detected_recovery_cue": 0,
  "fraction_with_at_least_one_detected_recovery_cue": 0.0,
  "h3_binary_baselines": {
    "action_text_only": {
      "view": "action_text_only",
      "horizon": "3",
      "task": "binary",
      "accuracy_mean": 0.5632615920240545,
      "accuracy_sd": 0.001408250651254712,
      "balanced_accuracy_mean": 0.6123092868988391,
      "balanced_accuracy_sd": 0.002220082476431209,
      "macro_f1_m

In [11]:
# 11. Mirror only this new audit folder to ir_final and verify SHA-256
DEST = IR_FINAL / OUT.name
DEST.mkdir(parents=True, exist_ok=True)

for src in OUT.iterdir():
    if src.is_file():
        shutil.copy2(src, DEST / src.name)

def folder_hashes(root):
    return {
        str(p.relative_to(root)): sha256_file(p)
        for p in sorted(root.rglob("*")) if p.is_file()
    }

source_hashes = folder_hashes(OUT)
mirror_hashes = folder_hashes(DEST)
assert source_hashes == mirror_hashes, "Mirror verification failed."

print("Source files:", len(source_hashes))
print("Mirror files:", len(mirror_hashes))
print("Mirror verification: True")
print("Source:", OUT)
print("Mirror:", DEST)


Source files: 6
Mirror files: 6
Mirror verification: True
Source: /content/drive/MyDrive/ir/NB8_mini_forensic_audit_20261124
Mirror: /content/drive/MyDrive/ir_final/NB8_mini_forensic_audit_20261124


In [12]:

# ============================================================
# FORENSIC DIAGNOSTIC — INSPECT H3 STATE TEXT BY LABEL
# ============================================================

from collections import defaultdict

examples_by_label = defaultdict(list)

for record_id, rec in primary_by_id.items():
    label = horizon_labels(rec)["3"]

    if len(examples_by_label[label]) < 3:
        examples_by_label[label].append(rec)

for label in ("reversible", "partially_reversible", "irreversible"):
    print("\n" + "=" * 90)
    print("H3 LABEL:", label)
    print("=" * 90)

    for i, rec in enumerate(examples_by_label[label], 1):
        print(f"\n--- Example {i} ---")
        print("Record ID:", rec.get("record_id"))
        print("\nSTATE TEXT:")
        print(rec.get("state_text", "<MISSING>"))
        print("\nACTION TEXT:")
        print(rec.get("action_text", "<MISSING>"))
        print("\n" + "-" * 70)



H3 LABEL: reversible

--- Example 1 ---
Record ID: None

STATE TEXT:
Simulated edit operation. Target=object-00000; original_value=original-00000; changed_value=changed-00000; amount=100; version_history_available=True.

ACTION TEXT:
Edit object-00000 to changed-00000

----------------------------------------------------------------------

--- Example 2 ---
Record ID: None

STATE TEXT:
Simulated delete operation. Target=object-00001; original_value=original-00001; changed_value=changed-00001; amount=125; recycle_bin_enabled=True.

ACTION TEXT:
Delete object-00001

----------------------------------------------------------------------

--- Example 3 ---
Record ID: None

STATE TEXT:
Simulated email operation. Target=object-00002; original_value=original-00002; changed_value=changed-00002; amount=150; recall_window_open=True.

ACTION TEXT:
Send simulated email for object-00002

----------------------------------------------------------------------

H3 LABEL: partially_reversible

--- Exa

In [13]:

# ============================================================
# FORENSIC CHECK — RECOVERY CUE DETECTION AND MASKING
# ============================================================

import re

patterns = {
    "version_history_available": r"version_history_available\s*=\s*(true|false)",
    "recycle_bin_enabled": r"recycle_bin_enabled\s*=\s*(true|false)",
    "recall_window_open": r"recall_window_open\s*=\s*(true|false)",
    "refund_eligible": r"refund_eligible\s*=\s*(true|false)",
    "transfer_status": r"transfer_status\s*=\s*\w+",
}

print("RECOVERY CUE MATCH COUNTS")
print("=" * 70)

for name, pattern in patterns.items():
    count = sum(
        bool(re.search(pattern, str(rec.get("state_text", "")), re.I))
        for rec in primary_by_id.values()
    )
    print(f"{name}: {count}/2000")

print("\nMASKING CHECK")
print("=" * 70)

for label in ("reversible", "partially_reversible", "irreversible"):
    rec = examples_by_label[label][0]
    original = str(rec.get("state_text", ""))

    masked = original
    for pattern in patterns.values():
        masked = re.sub(pattern, "[RECOVERY_CUE_MASKED]", masked, flags=re.I)

    print(f"\n{label.upper()}")
    print("ORIGINAL:", original)
    print("MASKED:  ", masked)


RECOVERY CUE MATCH COUNTS
version_history_available: 500/2000
recycle_bin_enabled: 500/2000
recall_window_open: 500/2000
refund_eligible: 500/2000
transfer_status: 500/2000

MASKING CHECK

REVERSIBLE
ORIGINAL: Simulated edit operation. Target=object-00000; original_value=original-00000; changed_value=changed-00000; amount=100; version_history_available=True.
MASKED:   Simulated edit operation. Target=object-00000; original_value=original-00000; changed_value=changed-00000; amount=100; [RECOVERY_CUE_MASKED].

PARTIALLY_REVERSIBLE
ORIGINAL: Simulated transfer operation. Target=object-00003; original_value=original-00003; changed_value=changed-00003; amount=175; refund_eligible=True; transfer_status=settled.
MASKED:   Simulated transfer operation. Target=object-00003; original_value=original-00003; changed_value=changed-00003; amount=175; [RECOVERY_CUE_MASKED]; [RECOVERY_CUE_MASKED].

IRREVERSIBLE
ORIGINAL: Simulated edit operation. Target=object-00000; original_value=original-00000; chan

In [14]:

# ============================================================
# VERIFY THE ACTUAL TEXT VIEWS USED BY THE BASELINE
# ============================================================

sample_id = next(iter(primary_by_id))
sample_record = primary_by_id[sample_id]

print("Sample ID:", sample_id)
print("\nOriginal state text:")
print(sample_record["state_text"])

print("\nState text view:")
print(TEXT_VIEWS["state_text_only"][sample_id])

print("\nMasked state text view:")
print(TEXT_VIEWS["masked_state_text_only"][sample_id])

print("\nAre state and masked views identical?",
      TEXT_VIEWS["state_text_only"][sample_id] ==
      TEXT_VIEWS["masked_state_text_only"][sample_id])

print("\nAre any recovery cues still present in masked view?")
masked_text = TEXT_VIEWS["masked_state_text_only"][sample_id]

for name, pattern in patterns.items():
    print(name, bool(re.search(pattern, masked_text, re.I)))


Sample ID: sbx-v2-row-00000-0

Original state text:
Simulated edit operation. Target=object-00000; original_value=original-00000; changed_value=changed-00000; amount=100; version_history_available=True.

State text view:
Simulated edit operation. Target=object-00000; original_value=original-00000; changed_value=changed-00000; amount=100; version_history_available=True.

Masked state text view:
Simulated edit operation. Target=object-00000; original_value=original-00000; changed_value=changed-00000; amount=100; [RECOVERY_CUE_MASKED].

Are state and masked views identical? False

Are any recovery cues still present in masked view?
version_history_available False
recycle_bin_enabled False
recall_window_open False
refund_eligible False
transfer_status False


## What this mini notebook intentionally does not do

- It does not rerun the 2,232 hidden-state probe configurations.
- It does not load Qwen or Llama.
- It does not modify NB4 data, frozen splits, NB7 shards, or original NB8 results.
- It does not investigate the separate 2,052-row legacy layer-0 collision report.
- It does not establish causal attribution from text-baseline scores to hidden-state probe scores.

If the lexical audit confirms strong explicit-cue predictability, the next focused experiment should be a **cue-masked hidden-state probe audit** on a small set of pre-specified layers, using the same frozen splits. That is much smaller than rerunning the entire NB8 sweep.